**Importing Modules**

In [7]:
import numpy as np
import pandas as pd

**Data Cleaning**

In [8]:
df=pd.read_csv("../data/Student Social Media And Mental Health Impact.csv")

In [9]:
df.drop_duplicates(inplace=True)

In [10]:
df["Physical_Activity_Hours"]=df["Physical_Activity_Hours"].clip(lower=0)

In [11]:
df.describe()

,Age,Avg_Daily_Usage_Hours,Daily_Unlocks,Study_Hours,Physical_Activity_Hours,Sleep_Hours_Per_Night,Mental_Health_Score
count,4998.000000,4998.000000,4998.000000,4998.000000,4998.000000,4998.000000,4998.000000
mean,20.822129,5.078491,171.455582,3.008403,1.751160,6.634654,6.231152
std,1.736774,1.654097,42.859829,1.636831,0.667282,1.221561,1.278476
min,18.000000,1.000000,62.000000,0.300000,0.000000,3.600000,3.600000
25%,19.000000,3.800000,140.000000,1.500000,1.300000,5.600000,5.100000
50%,21.000000,5.000000,171.000000,2.800000,1.700000,6.600000,6.100000
75%,22.000000,6.300000,204.000000,4.200000,2.200000,7.500000,7.100000
max,24.000000,8.800000,273.000000,8.300000,4.100000,9.900000,9.400000


In [12]:
numeric_columns=df.select_dtypes(include="number")
numeric_columns.skew()

Age                        0.155008
Avg_Daily_Usage_Hours      0.005575
Daily_Unlocks              0.002309
Study_Hours                0.436125
Physical_Activity_Hours    0.053288
Sleep_Hours_Per_Night      0.123919
Mental_Health_Score        0.207086
dtype: float64

**Feature Engineering**

In [13]:
common_countries=list(df['Country'].value_counts().index[:10])

def get_country(country):
    if country not in common_countries:
        return "Other"
    else:
        return country

df["Country"]=df["Country"].apply(get_country)
df["Country"].value_counts()

Country
Other        3231
India         389
USA           354
Canada        230
Australia     198
UK            185
Germany       136
Mexico         94
Turkey         94
France         87
Name: count, dtype: int64

In [14]:
df.select_dtypes(include="object").columns

Index(['Gender', 'Country', 'Academic_Level', 'Most_Used_Platform',
       'Purpose_Of_Use', 'Stress_Level'],
      dtype='object')

**Encoding Strategy**


In [15]:
skewd_col=["Study_Hours"]
other_col=["Age","Avg_Daily_Usage_Hours","Daily_Unlocks","Physical_Activity_Hours","Sleep_Hours_Per_Night"]
label_column=["Stress_Level"]
ohe_column=["Gender","Country","Academic_Level","Most_Used_Platform","Purpose_Of_Use"]

feature_columns=skewd_col+other_col+label_column+ohe_column
X=df[feature_columns]
y=df["Mental_Health_Score"]


**Building Inner Pipelines And Applying Column Transformation**



1) In stress level column the hierarchy has to be maintained so we use oe or le.

2) label encoder and ordinal encoder are similar but in le there is a chance in which order of the categories in a categorical column change. For eg stres level: high can be represented as 0 and low as 4 so it is recommended to use ordinal encoding

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer,StandardScaler,OrdinalEncoder,OneHotEncoder

skew_pipeline=Pipeline([('log_transform', FunctionTransformer(np.log1p)), ('scale', StandardScaler())])

other_col_pipeline=Pipeline([('scale', StandardScaler())])

label_column_pipeline=Pipeline([('encoding', OrdinalEncoder(categories=[["Low","Medium","High","Very High"]]))])

ohe_column_pipeline=Pipeline([('encoding', OneHotEncoder(handle_unknown='ignore'))])


**Each of your pipelines processes only one group of columns.**

Now we have created the pipelines but the pipeline don't know on which columns we have to apply the pipeline fuctions so we will use **Column Transformer** to apply the pipeline functions to the columns.

**ColumnTransformer(transformers=[(which pipeline,which feature)])**

In [17]:
from sklearn.compose import ColumnTransformer

Columntransform=ColumnTransformer(
    transformers=[
        ("Skewed_Pipeline",skew_pipeline,skewd_col),
        ("Other_Col_Pipeline",other_col_pipeline,other_col),
        ("Label_Col_Pipeline",label_column_pipeline,label_column),   
        ("Ohe_Col_Pipeline",ohe_column_pipeline,ohe_column)
        ])

```mermaid
flowchart LR

    A[Dataset] --> B[ColumnTransformer]

    B --> C1[Skewed Columns]
    C1 --> D1[Logarithmic Transform]
    D1 --> E1[StandardScaler]

    B --> C2[Other Numeric]
    C2 --> D2[StandardScaler]

    B --> C3[Ordinal Column]
    C3 --> D3[OrdinalEncoder]

    B --> C4[Nominal Columns or Ohe Columns]
    C4 --> D4[OneHotEncoder]

    E1 --> F[Combine]
    D2 --> F
    D3 --> F
    D4 --> F

    F --> G[Processed Dataset]
```

**Training And Testing Split Dataset**

We preprocess using the training data first to prevent data leakage and ensure that the model is evaluated on truly unseen data.

Why?

Many preprocessing techniques learn information from the data.

Examples:

1) StandardScaler learns mean and standard deviation.
2) MinMaxScaler learns minimum and maximum.
3) SimpleImputer learns mean/median/mode.
4) OrdinalEncoder and OneHotEncoder learn categories.
5) PCA learns principal components.

If these are fitted on the entire dataset (training + test), the model indirectly gets information from the test set before evaluation.

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

**Building Final/Outer Pipeline**

This pipeline chains the entire preprocessing step with the machine learning model.

The inner pipelines preprocess different columns. The outer pipeline combines the entire preprocessing step and the machine learning model into a single object, so you only call fit() and predict() once.

```mermaid
flowchart TD

A[Raw Dataset]
    --> B[Pipeline]

B --> C[ColumnTransformer]

C --> D1[Skew Pipeline]
C --> D2[Numeric Pipeline]
C --> D3[Ordinal Pipeline]
C --> D4[OneHot Pipeline]

D1 --> E[Combined Features]
D2 --> E
D3 --> E
D4 --> E

E --> F[Machine Learning Model]

F --> G[Prediction]
```

**Creating A Baseling Algorithm And Applying Outer Pipeline** : Linear Regression

In [19]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

model=LinearRegression()

outer_pipeline_lr=Pipeline([
    ("preprocessing",Columntransform),
    ("regressor",model)
])

outer_pipeline_lr.fit(X_train,y_train)
model_prediction=outer_pipeline_lr.predict(X_test)
model_prediction_training=outer_pipeline_lr.predict(X_train)

print(f"Accuracy of training : {r2_score(y_train,model_prediction_training)}")
print(f"Accuracy of testing : {r2_score(y_test,model_prediction)}")
print(f"MAE : {mean_absolute_error(y_test,model_prediction)}")
print(f"MSE : {mean_squared_error(y_test,model_prediction)}")


Accuracy of training : 0.7236771199387539
Accuracy of testing : 0.7397944617433021
MAE : 0.5361775634720182
MSE : 0.45701905273487986


**We calculate both training and testing accuracy (or R² for regression) to understand how well the model learns and how well it generalizes to unseen data.**


| Training R² | Testing R² | Interpretation |
|-------------|------------|----------------|
| High | High | ✅ Good model (generalizes well to unseen data). |
| High | Low | ❌ Overfitting (learned the training data too well but performs poorly on new data). |
| Low | Low | ❌ Underfitting (failed to capture the underlying patterns in the data). |
| Low | High | ⚠️ Unusual scenario; may indicate data leakage, randomness, incorrect train/test split, or a very small dataset. |

**Rule of thumb**
Training R² tells you how well the model learned.
Testing R² tells you how well the model will perform in the real world.

**A good model typically has:**

1) High training score,
2) High testing score, and
3) A small gap between the two scores.

**1) Difference < 0.05 → Excellent generalization.**


**2) Difference ≈ 0.05–0.15 → Mild/moderate overfitting (often acceptable).**


**3) Difference > 0.15–0.20 → Likely significant overfitting.**

**Training and testing using a stronger model like Random Forest**

A baseline model establishes a benchmark. A stronger model should only be chosen if it provides a meaningful improvement over that benchmark.

In [20]:
from sklearn.ensemble import RandomForestRegressor

model_rfc=RandomForestRegressor()

Outer_pipeline_rf=Pipeline([
    ("Preprocessing",Columntransform),
    ("random-forest",model_rfc)
    ])

Outer_pipeline_rf.fit(X_train,y_train)
model_rfc_prediction_train=Outer_pipeline_rf.predict(X_train)
model_rfc_prediction=Outer_pipeline_rf.predict(X_test)

print(f"Accuracy of training : {r2_score(y_train,model_rfc_prediction_train)}")
print(f"Accuracy of testing : {r2_score(y_test,model_rfc_prediction)}")
print(f"MAE : {mean_absolute_error(y_test,model_rfc_prediction)}")
print(f"MSE : {mean_squared_error(y_test,model_rfc_prediction)}")


Accuracy of training : 0.9812517300342883
Accuracy of testing : 0.8770268241145567
MAE : 0.3483909047619048
MSE : 0.215987271952381


Here the overall model has improved the result as compared to the baseline model but the result is being overfitted as there is accuracy diff bw the training and testing data so in order to prevent overfitting now we will do **Hyperparameter Tuning**.

In [21]:
from sklearn.model_selection import RandomizedSearchCV

hyperparameter_tuning = RandomizedSearchCV(
    estimator=Outer_pipeline_rf,

    param_distributions={
        "random-forest__n_estimators": [100, 200, 300],
        "random-forest__max_depth": [5, 10, 15],
        "random-forest__min_samples_split": [2, 5, 7],
        "random-forest__min_samples_leaf": [1, 2, 4]
    },

    n_iter=15,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

hyperparameter_tuning.fit(X_train, y_train)
print(hyperparameter_tuning.best_params_)
print(hyperparameter_tuning.best_score_)

{'random-forest__n_estimators': 200, 'random-forest__min_samples_split': 5, 'random-forest__min_samples_leaf': 2, 'random-forest__max_depth': 15}
0.8398400580654407


In [22]:
best_model=hyperparameter_tuning.best_estimator_
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)
print(f"Accuracy of testing : {r2_score(y_test,y_pred_test)}")

Accuracy of testing : 0.8658440771133397


**Since the accuracy score is decreasing after hyperparameter tuning so instead of using the tuned model it would be a better approach to use the model without tuning**

**Model Evaluation**

In [23]:
lrm_train_r2=r2_score(y_train,model_prediction_training)
lrm_test_r2=r2_score(y_test,model_prediction)
lrm_MAE=mean_absolute_error(y_test,model_prediction)
lrm_MSE=mean_squared_error(y_test,model_prediction)

rfc_train_r2=r2_score(y_train,model_rfc_prediction_train)
rfc_test_r2=r2_score(y_test,model_rfc_prediction)
rfc_MAE=mean_absolute_error(y_test,model_rfc_prediction)
rfc_MSE=mean_squared_error(y_test,model_rfc_prediction)

rfc_tuned_train_r2=r2_score(y_train,y_pred_train)
rfc_tuned_test_r2=r2_score(y_test,y_pred_test)
rfc_tuned_MAE=mean_absolute_error(y_test,y_pred_test)
rfc_tuned_MSE=mean_squared_error(y_test,y_pred_test)

evaluation_data=pd.DataFrame(
    {
        "model":['Linear model',"random forest","tuned random forest"],
        "R2_score_test":[lrm_test_r2,rfc_test_r2,rfc_tuned_test_r2],
        "R2_score_train":[lrm_train_r2,rfc_train_r2,rfc_tuned_train_r2],
        "MAE":[lrm_MAE,rfc_MAE,rfc_tuned_MAE],
        "MSE":[lrm_MSE,rfc_MSE,rfc_tuned_MSE]
    }
)
evaluation_data



,model,R2_score_test,R2_score_train,MAE,MSE
0,Linear model,0.739794,0.723677,0.536178,0.457019
1,random forest,0.877027,0.981252,0.348391,0.215987
2,tuned random forest,0.865844,0.955942,0.367420,0.235628


**Testing accuracy matters the most for evaluating a model**

**Saving the model**

In [24]:
import joblib

joblib.dump(Outer_pipeline_rf,"../models/model.pkl")
print("model saved.......")

model saved.......


In [ ]:
for i in df.columns.tolist():
    print(df[i].unique())